# Week 1 · Day 3 Lab — Convolution, filtering and histograms
**AIMS Cameroon · Introduction to Computer Vision**

You will implement the two workhorses of image processing yourself, **convolution** and **histogram equalisation**, and then use library versions with understanding.

**Learning goals**
1. Predict convolution output sizes.
2. Implement `conv2d` with padding and stride (loops, then vectorised) and verify it.
3. Run a denoising shoot-out and report PSNR.
4. Implement histogram equalisation from the CDF; compare with OpenCV; apply CLAHE.

In [ ]:
# If you are on Google Colab or Kaggle, everything below is pre-installed.
# Locally:  pip install numpy matplotlib scipy scikit-image scikit-learn opencv-python
import numpy as np
import matplotlib.pyplot as plt
import cv2, skimage, scipy
from skimage import data, color, util, exposure, filters, feature, measure, transform
from scipy import ndimage as ndi

plt.rcParams["figure.dpi"] = 100
rng = np.random.default_rng(0)

def show(*images, titles=None, cmap="gray", size=4, **kw):
    """Display images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(size * n, size))
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, images)):
        ax.imshow(im, cmap=cmap if np.ndim(im) == 2 else None, **kw)
        ax.axis("off")
        if titles:
            ax.set_title(titles[i])
    plt.tight_layout(); plt.show()

print("numpy", np.__version__, "| opencv", cv2.__version__, "| scikit-image", skimage.__version__)

In [ ]:
cam = data.camera()
def psnr(ref, img):
    mse = np.mean((ref.astype(float) - img.astype(float)) ** 2)
    return 10 * np.log10(255 ** 2 / mse)

## 1. Output size
### ✏️ Exercise 1.1
Implement $O = \lfloor (N+2p-K)/s \rfloor + 1$.

In [ ]:
def output_size(N, K, p=0, s=1):
    # TODO
    raise NotImplementedError

In [ ]:
assert output_size(32, 5, 2, 1) == 32      # "same"
assert output_size(224, 7, 3, 2) == 112    # ResNet stem
assert output_size(28, 3) == 26            # "valid"
print("✓ output_size passed")

## 2. Convolution (correlation) from scratch
We implement **cross-correlation**, which is what deep learning libraries call "convolution":
$$ g[x,y] = \sum_{i,j} h[i,j]\, I[x\cdot s+i,\; y\cdot s+j] \quad\text{(on the padded image)} $$

### ✏️ Exercise 2.1 — the loop version
`conv2d_loops(img, k, padding=0, stride=1)` for a 2D float image and a 2D kernel, with zero padding.

In [ ]:
def conv2d_loops(img, k, padding=0, stride=1):
    # TODO: pad, compute output size, slide the kernel, multiply and sum
    raise NotImplementedError

### ✏️ Exercise 2.2 — the vectorised version
Use `np.lib.stride_tricks.sliding_window_view` to get all patches at once, then `np.einsum` (or `tensordot`) to apply the kernel. Subsample the result for stride > 1.

In [ ]:
def conv2d(img, k, padding=0, stride=1):
    # TODO
    raise NotImplementedError

In [ ]:
small = cam[200:260, 200:270].astype(float)
k = rng.normal(size=(3, 3))
ref = ndi.correlate(small, k, mode="constant")             # zero padding, "same"
assert np.allclose(conv2d_loops(small, k, padding=1), ref)
assert np.allclose(conv2d(small, k, padding=1), ref)
assert conv2d(small, k, padding=1, stride=2).shape == (output_size(60, 3, 1, 2), output_size(70, 3, 1, 2))
print("✓ both conv2d versions match scipy.ndimage.correlate")

### 2.3 Worked example from the slides
The 5×5 patch with a bright centre, filtered by a 3×3 box kernel: the centre output should be 54.4.

In [ ]:
P = np.array([[10]*5, [10, 50, 50, 50, 10], [10, 50, 90, 50, 10], [10, 50, 50, 50, 10], [10]*5], float)
print(conv2d(P, np.ones((3, 3)) / 9)[1, 1].round(1))

### ✏️ Exercise 2.4 — speed and separability
Time `conv2d_loops`, `conv2d`, `cv2.filter2D` on the full camera image with a 7×7 Gaussian. Then verify that a 2D Gaussian equals two 1D passes (columns then rows) and time that too. Report a small table.

In [ ]:
# TODO

## 3. Denoising shoot-out
### ✏️ Exercise 3.1
For **Gaussian** noise (`var=0.01`) and **salt & pepper** noise (`amount=0.08`), apply: box 5×5, Gaussian σ=1.5, median 5×5, bilateral (`d=9, sigmaColor=75, sigmaSpace=75`). Fill the dictionary `results[(noise, filter)] = psnr` and print a table. Which filter wins for which noise, and why?

In [ ]:
noisy = {
    "gaussian": util.img_as_ubyte(util.random_noise(cam, mode="gaussian", var=0.01, rng=1)),
    "salt&pepper": util.img_as_ubyte(util.random_noise(cam, mode="s&p", amount=0.08, rng=2)),
}

In [ ]:
results = {}
# TODO

### ✏️ Exercise 3.2 — the median on paper
A 3×3 patch is all 128 except the centre (255). Compute the box and median outputs at the centre by hand, then check with NumPy.

In [ ]:
# TODO

## 4. Histogram equalisation from the CDF
$$ s_k = \mathrm{round}\Big((L-1)\sum_{j\le k} \frac{n_j}{n}\Big) $$

### ✏️ Exercise 4.1
1. `equalize_lut(counts, L)` returns the mapping $s_k$ for a histogram `counts` of length $L$.
2. Check it on the worked example `counts = [8, 6, 10, 0, 14, 2, 18, 6]` (expected `[1, 2, 3, 3, 4, 4, 6, 7]`).
3. `equalize(img)` applies it to a `uint8` image (use `np.bincount(img.ravel(), minlength=256)` and index the LUT with the image).

In [ ]:
def equalize_lut(counts, L):
    # TODO
    raise NotImplementedError

def equalize(img):
    # TODO
    raise NotImplementedError

In [ ]:
assert list(equalize_lut([8, 6, 10, 0, 14, 2, 18, 6], 8)) == [1, 2, 3, 3, 4, 4, 6, 7]
moon = data.moon()
mine, ocv = equalize(moon), cv2.equalizeHist(moon)
print("mean |mine - OpenCV| =", np.abs(mine.astype(int) - ocv).mean().round(2), "grey levels")
print("(OpenCV subtracts cdf_min first so the darkest level maps to 0; that is the only difference)")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(moon)
show(moon, mine, clahe, titles=["original", "my equalize", "CLAHE"], size=3.5)
fig, ax = plt.subplots(1, 3, figsize=(12, 2.5))
for a, im in zip(ax, [moon, mine, clahe]):
    a.hist(im.ravel(), 256, (0, 256)); a.set_yticks([])
plt.show()

### ✏️ Exercise 4.2 — colour images
Equalise `data.astronaut()` (a) per R, G, B channel and (b) on the **Y** channel of YCrCb only. Display both. Which one changes the hues? Explain.

In [ ]:
# TODO

## 5. Stretch
1. **Unsharp masking:** implement `sharpen(img, sigma, alpha)` $= I + \alpha(I - G_\sigma * I)$ (work in float, clip back to uint8). What happens to noise?
2. **Fourier filtering:** use `np.fft.fft2`/`fftshift` to keep only frequencies within radius 30 of the centre (low-pass) and the complement (high-pass). Relate the results to blurring and edge detection.

In [ ]:
# TODO